# WP 15 · Haar Cascade Face Detection
*Detection — Unit II: UAV Data Collection & Processing Methods*

The Viola-Jones Haar cascade (2001) made real-time detection possible on plain CPUs - still useful on compute-starved UAV boards. It uses rectangle features, the integral image, and a cascade that rejects boring windows early.

**Supporting data:** none — the sample image ships in `scikit-image` / OpenCV.

---
**How to use:** `Runtime ▸ Run all`, or run each cell top to bottom. The original teaching snippet is reproduced verbatim below; only GUI-only calls (`cv2.imshow`, `cv2.waitKey`, `cv2.destroyAllWindows`) are adapted, because Colab has no display window.

In [ ]:
# === Setup (run me first) ===================================================
# OpenCV, NumPy and Matplotlib are already installed in Google Colab.
# If you run locally and cv2 is missing, uncomment the next line:
# !pip install opencv-python-headless matplotlib

import cv2, numpy as np, os
import matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

def show(*imgs, titles=None, cmap=None, figsize=(13, 5)):
    """Display 1..N images inline. BGR images are auto-converted to RGB.
    (Colab has no window server, so cv2.imshow() cannot be used.)"""
    titles = titles or [""] * len(imgs)
    plt.figure(figsize=figsize)
    for i, im in enumerate(imgs):
        ax = plt.subplot(1, len(imgs), i + 1)
        if im.ndim == 2:
            ax.imshow(im, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(titles[i], fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# --- Get a test frame with a face -------------------------------------------
# 1) Upload YOUR OWN photo (any group/selfie) when prompted, OR
# 2) press Cancel to use a bundled public-domain image (skimage 'astronaut').
def get_face_frame():
    try:
        from google.colab import files
        print("Upload a photo containing faces (or Cancel for the sample image).")
        up = files.upload()
        for fn in up:
            im = cv2.imread(fn)
            if im is not None:
                print("Loaded", fn); return im
    except Exception:
        pass
    from skimage import data                       # preinstalled in Colab
    print("Using public-domain sample image (astronaut).")
    return cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)

frame = get_face_frame()


### The snippet, exactly as shown in the playground
```python
face = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
hits = face.detectMultiScale(gray,
        scaleFactor=1.1,     # image shrink step between scales
        minNeighbors=5,      # overlapping hits needed -> fewer false +
        minSize=(30, 30))

for (x, y, w, h) in hits:
    cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
```

In [ ]:
# The cascade XML ships INSIDE OpenCV - no download needed.
face = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
hits = face.detectMultiScale(gray,
        scaleFactor=1.1,     # image shrink step between scales
        minNeighbors=5,      # overlapping hits needed -> fewer false +
        minSize=(30, 30))

vis = frame.copy()
for (x, y, w, h) in hits:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 0, 0), 2)

print(f"Detected {len(hits)} face(s)")
show(frame, vis, titles=["Input frame", f"{len(hits)} detection(s)"])

In [ ]:
# --- Tune the two dials and watch precision/recall trade off ----------------
for mn in (3, 5, 8):
    h = face.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=mn, minSize=(30, 30))
    print(f"minNeighbors={mn} -> {len(h)} hits (higher = fewer false positives)")
# scaleFactor closer to 1.0 finds more sizes but runs slower;
# larger (1.3) is faster but may skip faces.